In [ ]:
%pip install -q langchain-classic langgraph langchain-openai langchain-core langchain-community langchain-text-splitters faiss-cpu "langchain-chroma>=0.1.2" langchain-elasticsearch
!pip install  pypdf unstructured unstructured[docx] unstructured[pdf] langchain-docling langchain_unstructured

In [ ]:
import os
with open("/content/api_key.txt") as archivo:
  apikey = archivo.read().strip()
  os.environ["OPENAI_API_KEY"] =apikey

In [ ]:
import os
with open("/content/elasticstore.txt") as archivo:
  key_elastic = archivo.read().strip()

In [ ]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import WebBaseLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_elasticsearch import ElasticsearchStore

In [ ]:
loader = WebBaseLoader(
    web_paths=("https://wpvet.com/exotic-pets-care-guides/getting-a-reptile-as-a-pet/",
    "https://wpvet.com/exotic-pets-care-guides/reptile-shedding-ecdysis/",
    "https://wpvet.com/exotic-pets-care-guides/reptile-diseases/",
    "https://wpvet.com/exotic-pets-care-guides/reptile-gut-loading/",
    "https://wpvet.com/exotic-pets-care-guides/leopard-geckos-african-fat-tails-geckos/",
    "https://wpvet.com/exotic-pets-care-guides/bearded-dragons/",
    "https://wpvet.com/exotic-pets-care-guides/green-water-dragons/",
    "https://wpvet.com/exotic-pets-care-guides/green-iguana-care-guide/",
    "https://wpvet.com/exotic-pets-care-guides/green-iguana-diet/",
    "https://wpvet.com/exotic-pets-care-guides/corn-and-rat-snakes/",
    "https://wpvet.com/exotic-pets-care-guides/boas-pythons/",
    "https://wpvet.com/exotic-pets-care-guides/chameleons/",
    "https://wpvet.com/exotic-pets-care-guides/tortoises/",
    "https://wpvet.com/exotic-pets-care-guides/aquatic-turtles/",
    "https://wpvet.com/exotic-pets-care-guides/rat-care/",
    "https://wpvet.com/exotic-pets-care-guides/hamster-care/",
    "https://wpvet.com/exotic-pets-care-guides/hamster-diseases/",
    "https://wpvet.com/exotic-pets-care-guides/reptile-nutrition/",
    "https://wpvet.com/exotic-pets-care-guides/reptile-lighting/",
    "https://wpvet.com/exotic-pets-care-guides/reptile-temperature/"
))
docs_web = loader.load()

docs_pdf = []
for archivo in ["handbook.pdf", "exotic companion.pdf"]:
    loader_pdf = PyPDFLoader(archivo)
    docs_pdf.extend(loader_pdf.load())

docs_totales = docs_web + docs_pdf

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
all_splits = text_splitter.split_documents(docs_totales)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
db = ElasticsearchStore.from_documents(
    all_splits,
    embeddings,
    es_url="ELASTIC_URL",
    es_user="elastic",
    es_password=key_elastic,
    index_name="ELASTIC_INDEX")

db.client.indices.refresh(index="indx_app")

ObjectApiResponse({'_shards': {'total': 2, 'successful': 1, 'failed': 0}})

In [ ]:
vector_store = ElasticsearchStore(
    es_url="ELASTIC_URL",
    es_user="elastic",
    es_password=key_elastic,
    index_name="ELASTIC_INDEX",
    embedding=embeddings
    )

retriever_chain = vector_store.as_retriever(search_kwargs={"k": 4})

tool_04 = retriever_chain.as_tool(
    name="info_mascotas",
    description="Informacion detallada sobre el cuidado de mascotas exóticas",
)

/tmp/ipykernel_3727/546872877.py:11: LangChainBetaWarning: This API is in beta and may change in the future.
  tool_04 = retriever_chain.as_tool(
